In [1]:
import pandas as pd


test = pd.read_csv('spacecraft_test_preprocessed.csv')
train = pd.read_csv('spacecraft_train_preprocessed.csv')

In [4]:
train

tournament  game_length  winner  max_collection_rate  apm   spm  \
0            0.0          848       0                 3778  316  26.3   
1            1.0          612       1                 4344  398  42.9   
2            1.0          486       0                 3095  389  35.9   
3            0.0         1078       0                 4064  449  40.2   
4            2.0         1001       1                 4484  627  37.6   
...          ...          ...     ...                  ...  ...   ...   
1653         2.0         1059       0                 3471  415  33.9   
1654         1.0          473       0                 2625  410  16.5   
1655         1.0          650       1                 3510  335  45.0   
1656         0.0          881       0                 3862  379  45.5   
1657         1.0          447       1                 2311  296  45.5   

      workers_produced  workers_killed  workers_lost  supply_block  ...  \
0                   75              18            17            75  ...   
1                   97              22            21            30  ...   
2                   80              11            17            20  ...   
3                   77              68            30            15  ...   
4                   98              24            25            25  ...   
...                ...             ...           ...           ...  ...   
1653                89              54            51            45  ...   
1654                49               3             4            20  ...   
1655                66               7             3             5  ...   
1656               100              30            37            40  ...   
1657                48              18             6            50  ...   

      avg_pac_gap  avg_collection_rate_minerals  avg_unspent_minerals  \
0            0.06                        1808.2                 241.2   
1            0.06                        1938.9                 473.6   
2            0.06                        1563.7                 232.1   
3            0.06                        2107.6                 535.6   
4            0.04                        2306.8                 259.8   
...           ...                           ...                   ...   
1653         0.07                        1583.5                 341.4   
1654         0.04                        1246.9                 150.1   
1655         0.08                        1876.5                 268.4   
1656         0.08                        1622.3                 292.6   
1657         0.07                        1064.1                 122.5   

      minerals_lost  minerals_collected  avg_collection_rate_gas  \
0             10341               25162                    533.8   
1              3075               20305                    473.9   
2              4200               13525                    226.5   
3             13650               37470                    749.7   
4             21675               37048                    826.1   
...             ...                 ...                      ...   
1653          12968               27055                    446.9   
1654           1725               10705                    332.4   
1655           8575               21220                    457.3   
1656           5535               24609                    580.9   
1657            475                8575                    269.3   

      avg_unspent_gas  gas_lost  gas_collected     race  
0               273.8      3436           6384   Terran  
1               236.7       625           4265     Zerg  
2               157.6       800           1668     Zerg  
3               401.2      5850          11682  Protoss  
4               299.8      8975          11930     Zerg  
...               ...       ...            ...      ...  
1653            147.0      4018           6423     Zerg  
1654            129.8       600           2432  Protoss  
1655        

In [34]:
# train catboost and crossval using f1 macro

from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(iterations=1000, task_type='GPU', devices='0:1', verbose=100)
scores = cross_val_score(model, train.drop('race', axis=1), train['race'], cv=cv, scoring='f1_macro')


Learning rate set to 0.068752
0:	learn: 1.0561256	total: 7.5ms	remaining: 7.49s
100:	learn: 0.3534209	total: 651ms	remaining: 5.8s
200:	learn: 0.2300199	total: 1.46s	remaining: 5.79s
300:	learn: 0.1662798	total: 2.29s	remaining: 5.33s
400:	learn: 0.1285923	total: 3.05s	remaining: 4.56s
500:	learn: 0.1030930	total: 3.78s	remaining: 3.77s
600:	learn: 0.0850595	total: 4.62s	remaining: 3.07s
700:	learn: 0.0708116	total: 5.41s	remaining: 2.31s
800:	learn: 0.0592308	total: 6.18s	remaining: 1.53s
900:	learn: 0.0510268	total: 7.06s	remaining: 775ms
999:	learn: 0.0442083	total: 7.84s	remaining: 0us
Learning rate set to 0.068752
0:	learn: 1.0563850	total: 9.95ms	remaining: 9.94s
100:	learn: 0.3546707	total: 783ms	remaining: 6.97s
200:	learn: 0.2307530	total: 1.54s	remaining: 6.13s
300:	learn: 0.1660204	total: 2.41s	remaining: 5.6s
400:	learn: 0.1271202	total: 3.29s	remaining: 4.91s
500:	learn: 0.1003471	total: 4.08s	remaining: 4.06s
600:	learn: 0.0809690	total: 4.82s	remaining: 3.2s
700:	learn: 

In [35]:
# do a hold out validation

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(train.drop('race', axis=1), train['race'])

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

# print feature importance
feature_importance = pd.DataFrame({'feature': X_train.columns, 'importance': model.feature_importances_})


Learning rate set to 0.107491
0:	learn: 1.0326810	test: 1.0437166	best: 1.0437166 (0)	total: 8.73ms	remaining: 8.72s
100:	learn: 0.2620525	test: 0.4906309	best: 0.4906309 (100)	total: 741ms	remaining: 6.59s
200:	learn: 0.1522412	test: 0.4371475	best: 0.4371363 (199)	total: 1.47s	remaining: 5.84s
300:	learn: 0.1015554	test: 0.4172742	best: 0.4172742 (300)	total: 2.15s	remaining: 4.98s
400:	learn: 0.0721950	test: 0.4070750	best: 0.4070274 (396)	total: 2.87s	remaining: 4.29s
500:	learn: 0.0544801	test: 0.3999960	best: 0.3999960 (500)	total: 3.68s	remaining: 3.66s
600:	learn: 0.0428890	test: 0.3975910	best: 0.3969171 (551)	total: 4.41s	remaining: 2.93s
700:	learn: 0.0351369	test: 0.3977491	best: 0.3969171 (551)	total: 5.13s	remaining: 2.19s
800:	learn: 0.0295253	test: 0.3974242	best: 0.3963246 (723)	total: 5.91s	remaining: 1.47s
900:	learn: 0.0251468	test: 0.3989854	best: 0.3963246 (723)	total: 6.7s	remaining: 736ms
999:	learn: 0.0216213	test: 0.3983154	best: 0.3963246 (723)	total: 7.41s	r

In [36]:
feature_importance.sort_values('importance', ascending=False)

,feature,importance
4,apm,17.566814
6,workers_produced,8.310914
8,workers_lost,7.925402
5,spm,7.221382
14,avg_pac_gap,6.852394
21,gas_lost,4.863165
13,avg_pac_actions,4.646441
9,supply_block,4.620162
16,avg_unspent_minerals,4.281055
3,max_collection_rate,4.119681


In [37]:
importance_list = feature_importance.sort_values('importance', ascending=False)['feature'].tolist()

In [38]:
# do a feature selection, each loop pop the least important feature and train the model and do cross val

saved_scores = pd.DataFrame(columns=['feature', 'score'])

for i in range(1, len(importance_list)):
    print(f'Feature selection: {i}')
    
    # Calculate the cross-validation scores
    scores = cross_val_score(model, train.drop(['race'], axis=1)[importance_list[:i]], train['race'], cv=cv, scoring='f1_macro')
    
    # Create a new DataFrame with the feature subset and mean score
    new_row = pd.DataFrame({'feature': [importance_list[:i]], 'score': [scores.mean()]})
    
    # Concatenate the new row to the saved_scores DataFrame and update it
    saved_scores = pd.concat([saved_scores, new_row], ignore_index=True)
    
    # Print the mean score for the current feature selection
    print(scores.mean())

Feature selection: 1
Learning rate set to 0.068752
0:	learn: 1.0640714	total: 5.71ms	remaining: 5.71s
100:	learn: 0.7892613	total: 548ms	remaining: 4.88s
200:	learn: 0.7744272	total: 1.08s	remaining: 4.3s
300:	learn: 0.7667309	total: 1.56s	remaining: 3.63s
400:	learn: 0.7617366	total: 2.08s	remaining: 3.1s
500:	learn: 0.7579033	total: 2.6s	remaining: 2.59s
600:	learn: 0.7549318	total: 3.14s	remaining: 2.09s
700:	learn: 0.7526744	total: 3.71s	remaining: 1.58s
800:	learn: 0.7507212	total: 4.32s	remaining: 1.07s
900:	learn: 0.7491131	total: 4.89s	remaining: 537ms
999:	learn: 0.7480312	total: 5.46s	remaining: 0us
Learning rate set to 0.068752
0:	learn: 1.0642062	total: 7.76ms	remaining: 7.75s
100:	learn: 0.7882330	total: 532ms	remaining: 4.73s
200:	learn: 0.7722264	total: 1.12s	remaining: 4.46s
300:	learn: 0.7633991	total: 1.66s	remaining: 3.85s
400:	learn: 0.7575723	total: 2.21s	remaining: 3.29s
500:	learn: 0.7537690	total: 2.75s	remaining: 2.74s
600:	learn: 0.7507811	total: 3.29s	remaini

Learning rate set to 0.068752
0:	learn: 1.0625135	total: 7.49ms	remaining: 7.49s
100:	learn: 0.7046565	total: 579ms	remaining: 5.15s
200:	learn: 0.6585570	total: 1.17s	remaining: 4.64s
300:	learn: 0.6212400	total: 1.76s	remaining: 4.09s
400:	learn: 0.5888420	total: 2.4s	remaining: 3.58s
500:	learn: 0.5612481	total: 3.07s	remaining: 3.06s
600:	learn: 0.5374998	total: 3.67s	remaining: 2.44s
700:	learn: 0.5150250	total: 4.27s	remaining: 1.82s
800:	learn: 0.4956312	total: 4.87s	remaining: 1.21s
900:	learn: 0.4764819	total: 5.47s	remaining: 601ms
999:	learn: 0.4605015	total: 6.23s	remaining: 0us
Learning rate set to 0.068752
0:	learn: 1.0632998	total: 7.04ms	remaining: 7.04s
100:	learn: 0.7164097	total: 723ms	remaining: 6.43s
200:	learn: 0.6665393	total: 1.45s	remaining: 5.76s
300:	learn: 0.6287439	total: 2.21s	remaining: 5.13s
400:	learn: 0.5986984	total: 2.94s	remaining: 4.4s
500:	learn: 0.5718273	total: 3.61s	remaining: 3.6s
600:	learn: 0.5466673	total: 4.25s	remaining: 2.82s
700:	learn:

In [42]:
# print saved scores based on feature selection

saved_scores.sort_values('score', ascending=False)


,feature,score
20,"[apm, workers_produced, workers_lost, spm, avg...",0.856579
18,"[apm, workers_produced, workers_lost, spm, avg...",0.851260
21,"[apm, workers_produced, workers_lost, spm, avg...",0.850444
19,"[apm, workers_produced, workers_lost, spm, avg...",0.849223
17,"[apm, workers_produced, workers_lost, spm, avg...",0.848308
15,"[apm, workers_produced, workers_lost, spm, avg...",0.846196
14,"[apm, workers_produced, workers_lost, spm, avg...",0.844705
13,"[apm, workers_produced, workers_lost, spm, avg...",0.839785
16,"[apm, workers_produced, workers_lost, spm, avg...",0.836628
12,"[apm, workers_produced, workers_lost, spm, avg...",0.836217


In [39]:
# print the best feature selection as list
best_feature_selection = saved_scores.loc[saved_scores['score'].idxmax()]['feature']

best_feature_selection

['apm',
 'workers_produced',
 'workers_lost',
 'spm',
 'avg_pac_gap',
 'gas_lost',
 'avg_pac_actions',
 'supply_block',
 'avg_unspent_minerals',
 'max_collection_rate',
 'avg_collection_rate_gas',
 'game_length',
 'avg_pac_action_latency',
 'avg_unspent_gas',
 'sq',
 'avg_pac_per_min',
 'workers_killed',
 'tournament',
 'gas_collected',
 'avg_collection_rate_minerals',
 'minerals_lost']

In [26]:
pd.concat([saved_scores, pd.DataFrame({'feature': importance_list, 'score': scores})], axis=0).to_csv('feature_selection.csv', index=False)

AttributeError: 'DataFrame' object has no attribute 'concat'